### Notebook 1: Cleans JSON data and loads into DuckDb to perform feature engineering

In [1]:
import duckdb as dd

In [ ]:
# Create a persistent connection to duckdb to save data for use later. Extracted folder should be labelled the same, otherwise file path can be changed to what contains the json objects.
conn = dd.connect('spotify_all_years.db')

In [ ]:
#First table: combine all year's JSON files into one
conn.execute(
"CREATE TABLE complete_data AS " \
"SELECT * " \
"FROM read_json_auto('Spotify Extended Streaming History/*.json', union_by_name=true)"
)

In [ ]:
#Inspect data
conn.execute("SELECT * FROM complete_data LIMIT 5").df()

In [ ]:
# Check how many columns + non-null values 
conn.execute("SELECT * FROM complete_data").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 101657 entries, 0 to 101656
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 101657 non-null  datetime64[us]
 1   platform                           101657 non-null  str           
 2   ms_played                          101657 non-null  int64         
 3   conn_country                       101657 non-null  str           
 4   ip_addr                            101657 non-null  str           
 5   master_metadata_track_name         100994 non-null  str           
 6   master_metadata_album_artist_name  100994 non-null  str           
 7   master_metadata_album_album_name   100994 non-null  str           
 8   spotify_track_uri                  100994 non-null  str           
 9   episode_name                       597 non-null     str           
 10  episode_show_name              

In [ ]:
#remove columns we won't use and make a new table
conn.execute("CREATE TABLE clean_data AS SELECT ts,platform, ms_played, ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,reason_end,shuffle,skipped FROM complete_data")

In [ ]:
#verify columns were dropped
conn.execute("SELECT * FROM clean_data").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 101657 entries, 0 to 101656
Data columns (total 12 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 101657 non-null  datetime64[us]
 1   platform                           101657 non-null  str           
 2   ms_played                          101657 non-null  int64         
 3   ip_addr                            101657 non-null  str           
 4   master_metadata_track_name         100994 non-null  str           
 5   master_metadata_album_artist_name  100994 non-null  str           
 6   master_metadata_album_album_name   100994 non-null  str           
 7   spotify_track_uri                  100994 non-null  str           
 8   reason_start                       101657 non-null  str           
 9   reason_end                         101657 non-null  str           
 10  shuffle                        

In [11]:
# dicrepancy between # entries and tracks because podcasts are still counted.
# remove podcasts/episodes from entries, Ran against just one column for optimization.
conn.execute("CREATE TABLE podcast_dropped AS SELECT * FROM clean_data WHERE master_metadata_album_artist_name IS NOT NULL")
#conn.execute("CREATE TABLE podcast_dropped AS SELECT * FROM clean_data WHERE master_metadata_track_name IS NOT NULL AND master_metadata_album_artist_name IS NOT NULL AND master_metadata_album_album_name IS NOT NULL AND spotify_track_uri IS NOT NULL")
#Another way to perform the same query but would alter original table:
#conn.execute("DELETE FROM clean_data WHERE master_metadata_track_name IS NULL AND master_metadata_album_artist_name IS NULL AND master_metadata_album_album_name IS NULL AND spotify_track_uri IS NULL ")

In [3]:
#verify # of entries went down (meaning podcasts and episodes removed)
conn.execute("SELECT * FROM podcast_dropped").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 100994 entries, 0 to 100993
Data columns (total 12 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 100994 non-null  datetime64[us]
 1   platform                           100994 non-null  str           
 2   ms_played                          100994 non-null  int64         
 3   ip_addr                            100994 non-null  str           
 4   master_metadata_track_name         100994 non-null  str           
 5   master_metadata_album_artist_name  100994 non-null  str           
 6   master_metadata_album_album_name   100994 non-null  str           
 7   spotify_track_uri                  100994 non-null  str           
 8   reason_start                       100994 non-null  str           
 9   reason_end                         100994 non-null  str           
 10  shuffle                        

### Table 1 : Unique songs + unique albums per artist

In [ ]:
# num of diff songs and diff albums listened to by artist.
# NOT accurate includes released singles as albums.
conn.execute("CREATE TABLE unique_songs_and_artists_per_artist AS SELECT master_metadata_album_artist_name, COUNT(DISTINCT master_metadata_album_album_name) AS unique_albums_for_artist, COUNT(DISTINCT master_metadata_track_name) AS num_unique_songs_for_artist FROM podcast_dropped GROUP BY master_metadata_album_artist_name")

In [ ]:
# NOT USED: distinct tracks per album per artist
# Limitation: if a user listened to an artist but only 1 song of an album it will not be included because it is considering it a single
conn.execute("SELECT master_metadata_album_artist_name AS artist_name, master_metadata_album_album_name AS album_name, COUNT(DISTINCT master_metadata_track_name) AS num_tracks " \
"FROM podcast_dropped " \
"GROUP BY artist_name, album_name " \
"HAVING num_tracks > 1").df()

### Table 2: Count of non-single albums per artists

In [8]:
#One row per artist, with a count of only the albums where you listened to more than 1 track. Non-single albumbs per artist.
conn.execute("CREATE TABLE non_single_albums_per_artist AS" \
" SELECT master_metadata_album_artist_name, COUNT(album_name) AS num_true_albums" \
" FROM (SELECT master_metadata_album_artist_name, master_metadata_album_album_name AS album_name, COUNT(DISTINCT master_metadata_track_name) AS num_tracks FROM podcast_dropped GROUP BY master_metadata_album_artist_name, album_name HAVING num_tracks > 1) AS album_counts" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,487


### Table 3: Play count by artist

In [9]:
# Total play count per artist
conn.execute("CREATE TABLE play_count AS SELECT COUNT(*) AS play_count_per_artist,master_metadata_album_artist_name"\
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,2733


### Table 4: Listening Consistency

In [ ]:
# Old listening consistency — COUNT(DISTINCT month/year) per artist.
# artist_name | months_active (Jan2019, Feb2019, March2020 -> counted up)
conn.execute("CREATE TABLE listening_consistency AS SELECT master_metadata_album_artist_name, COUNT(DISTINCT strftime('%Y-%m', ts)) AS listening_consistency " \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,2733


In [ ]:
# New listening consistency — COUNT(DISTINCT month/year) per artist where distinct songs listened to were > 1.
# artist_name | months_active (Jan2019, Feb2019, March2020 -> counted up but where songs were > 1)
conn.execute("CREATE TABLE listening_consistency AS SELECT master_metadata_album_artist_name, COUNT(*) AS survived_year_months" \
" FROM (SELECT master_metadata_album_artist_name, strftime('%Y-%m', ts) AS year_month, COUNT(DISTINCT master_metadata_track_name) AS distinct_songs_listened" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name, strftime('%Y-%m', ts)" \
" HAVING distinct_songs_listened > 1) AS subquery" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,621


### Table 5: Skip Rate

In [11]:
# Skip rate per artist, higher = you skip the artist most of the time, lower = you let them play out.
conn.execute("CREATE TABLE skip_rate AS SELECT master_metadata_album_artist_name, AVG(CAST(skipped AS INT)) as skip_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()


,Count
0,2733


### Table 6: Intentional Listening Rate

In [12]:
# Intentional listening rate: percentage of plays where reason_start IN ('clickrow', 'playbtn', 'backbtn') per artist.
conn.execute("CREATE TABLE intentional_listening_rate AS SELECT master_metadata_album_artist_name, SUM(CAST(reason_start IN ('clickrow', 'playbtn', 'backbtn') AS INT)) / COUNT(*) AS intentional_listening_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,2733


### Table 7 : Completion Rate

In [13]:
# Completion rate — AVG(ms_played) per artist as a proxy (you don't have track duration, so average ms played is the best you can do).
conn.execute("CREATE TABLE completion_rate AS SELECT master_metadata_album_artist_name, SUM(CAST(reason_end = 'trackdone' AS INT)) / COUNT(*) AS completion_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df() 

,Count
0,2733


### Table 8: Revisit-Rate

In [ ]:
# Learning (Not used): Simple window function to show each row's play alongside total play count without a group by.
conn.execute("SELECT master_metadata_album_artist_name, COUNT(*) OVER (PARTITION BY master_metadata_album_artist_name) AS total_plays" \
" FROM podcast_dropped").df()

,master_metadata_album_artist_name,total_plays
0,Zay Hilfigerrr,52
1,gnash,83
2,The Weeknd,4077
3,Maroon 5,244
4,Charlie Puth,65
...,...,...
100989,Dreamville,11
100990,The Neighbourhood,3481
100991,The Neighbourhood,3481
100992,BETWEEN FRIENDS,385


In [26]:
conn.execute("CREATE TABLE revisit_rate AS SELECT master_metadata_album_artist_name, COUNT(*) AS survived_revists_after_120" \
" FROM (SELECT master_metadata_album_artist_name, ts, LAG(ts,1) OVER(PARTITION BY master_metadata_album_artist_name ORDER BY ts) AS prev_timestamp, DATEDIFF('day',prev_timestamp,ts) AS difference_days FROM podcast_dropped) AS all_revisits_difference" \
" WHERE difference_days > 120" \
" GROUP BY master_metadata_album_artist_name").df()

,Count
0,1106


## Final Feature Engineered Table

In [51]:
conn.execute("CREATE TABLE feature_engineered_table AS" \
" SELECT p.master_metadata_album_artist_name, p.play_count_per_artist, u.unique_albums_for_artist, u.num_unique_songs_for_artist, c.completion_rate, i.intentional_listening_rate, s.skip_rate,l.survived_year_months AS listening_consistency,n.num_true_albums, r.survived_revists_after_120 AS revisit_rate" \
" FROM play_count p" \
" LEFT JOIN unique_songs_and_artists_per_artist u ON p.master_metadata_album_artist_name = u.master_metadata_album_artist_name" \
" LEFT JOIN completion_rate c ON p.master_metadata_album_artist_name = c.master_metadata_album_artist_name" \
" LEFT JOIN intentional_listening_rate i ON p.master_metadata_album_artist_name = i.master_metadata_album_artist_name" \
" LEFT JOIN skip_rate s ON p.master_metadata_album_artist_name = s.master_metadata_album_artist_name" \
" LEFT JOIN listening_consistency l ON p.master_metadata_album_artist_name = l.master_metadata_album_artist_name" \
" LEFT JOIN non_single_albums_per_artist n ON p.master_metadata_album_artist_name=n.master_metadata_album_artist_name" \
" LEFT JOIN revisit_rate r ON p.master_metadata_album_artist_name=r.master_metadata_album_artist_name")

In [52]:
conn.execute("SELECT * FROM feature_engineered_table").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 2733 entries, 0 to 2732
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   master_metadata_album_artist_name  2733 non-null   str    
 1   play_count_per_artist              2733 non-null   int64  
 2   unique_albums_for_artist           2733 non-null   int64  
 3   num_unique_songs_for_artist        2733 non-null   int64  
 4   completion_rate                    2733 non-null   float64
 5   intentional_listening_rate         2733 non-null   float64
 6   skip_rate                          2733 non-null   float64
 7   listening_consistency              621 non-null    Int64  
 8   num_true_albums                    487 non-null    Int64  
 9   revisit_rate                       1106 non-null   Int64  
dtypes: Int64(3), float64(3), int64(3), str(1)
memory usage: 221.7 KB
